# 05. Feature Engineering Modularizado com Pipeline Anti-Leakage

## 🎯 Objetivo

Implementar **feature engineering avançado** usando o módulo `source.features` com:
- **Velocity Features**: Agregações temporais em janelas deslizantes
- **Ratio Features**: Comparações com histórico do usuário
- **Behavioral Features**: Desvios de padrão + Smurfing Detection
- **Zero Data Leakage**: Fit APENAS em treino, Transform em treino + OOT

## 📚 Features Implementadas

### 1. **Velocity Features** (Janelas Deslizantes)
- Contagem, soma, média e máximo em 1h, 24h, 7d
- Standard deviation em 7 dias
- Criado por: `VelocityFeatureGenerator`

### 2. **Ratio Features** (Histórico de 30 dias)
- Ratio: Valor atual / Média histórica
- Ratio: Valor atual / Máximo histórico  
- Z-score: Desvio em relação à média
- Criado por: `RatioFeatureGenerator`

### 3. **Behavioral Features**
- Tempo desde última transação (segundos)
- Mudança de banco (flag)
- Novo país (flag)
- Hora incomum (flag para fora do comercial)
- Criado por: `BehavioralFeatureGenerator`

### 4. **Smurfing Detection** 🚩
- Contagem de transações $8k-$10k em 24h
- Soma total de transações estruturadas
- Score de proximidade ao threshold
- Detecta tentativas de "estruturação"

## ⚠️ Garantias Anti-Leakage

✅ **Fit APENAS em treino**  
✅ **Transform em treino + OOT com MESMO pipeline**  
✅ **Janelas deslizantes excluem transação atual (closed='left')**  
✅ **Ordenação temporal validada**  
✅ **PSI < 0.1 para estabilidade entre treino e OOT**

---

## 1. Setup - Importações e Configuração

In [ ]:
# ===== SETUP DE PATH =====
# Adiciona o diretório pai ao path para garantir que 'source' seja encontrado
import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
sys.path.insert(0, str(project_root))

print(f"✅ Project root adicionado ao sys.path: {project_root}")
print(f"   Notebook directory: {notebook_dir}")

# ===== AUTORELOAD =====
# Carrega automaticamente mudanças nos módulos
%load_ext autoreload
%autoreload 2

print("✅ Autoreload configurado (autoreload 2)")

# ===== IMPORTS PADRÃO =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configurar estilo
sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 6)

# ===== IMPORTS DO source/ =====
from source.config import get_data_path
from source.features import (
    VelocityFeatureGenerator,
    RatioFeatureGenerator,
    BehavioralFeatureGenerator,
    FeatureEngineeringPipeline
)
from source.modeling.metrics import BusinessMetrics

print("✅ Todos os imports realizados com sucesso!")
print(f"\nMódulos carregados:")
print(f"  - VelocityFeatureGenerator")
print(f"  - RatioFeatureGenerator")
print(f"  - BehavioralFeatureGenerator")
print(f"  - FeatureEngineeringPipeline")
print(f"  - BusinessMetrics")

## 2. Carregamento dos Dados (Treino e OOT)

⚠️ **IMPORTANTE**: Carregamos os datasets **APÓS** a divisão temporal feita no Notebook 04.
Isso garante que o feature engineering seja aplicado de forma independente em cada conjunto.

In [ ]:
print("="*80)
print("CARREGAMENTO DOS DADOS")
print("="*80)

# Carregar datasets usando config.py
df_treino = pd.read_csv(get_data_path('df_treino.csv', 'processed'))
df_oot = pd.read_csv(get_data_path('df_oot.csv', 'processed'))

# Converter Timestamp para datetime (formato ISO8601)
df_treino['Timestamp'] = pd.to_datetime(df_treino['Timestamp'], format='ISO8601')
df_oot['Timestamp'] = pd.to_datetime(df_oot['Timestamp'], format='ISO8601')

print(f"\n📊 Dataset de Treino:")
print(f"   Shape: {df_treino.shape}")
print(f"   Período: {df_treino['Timestamp'].min()} até {df_treino['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_treino['Is Laundering'].mean()*100:.2f}%")

print(f"\n📊 Dataset de OOT:")
print(f"   Shape: {df_oot.shape}")
print(f"   Período: {df_oot['Timestamp'].min()} até {df_oot['Timestamp'].max()}")
print(f"   Taxa de lavagem: {df_oot['Is Laundering'].mean()*100:.2f}%")

print(f"\n✅ Dados carregados com sucesso!")

## 3. Validação da Ordenação Temporal

**CRÍTICO**: Dados DEVEM estar ordenados por Timestamp antes de calcular features temporais.

In [ ]:
# Verificar ordenação
print("="*80)
print("VALIDAÇÃO DA ORDENAÇÃO TEMPORAL")
print("="*80)

treino_sorted = df_treino['Timestamp'].is_monotonic_increasing
oot_sorted = df_oot['Timestamp'].is_monotonic_increasing

print(f"\n✓ Treino ordenado: {treino_sorted}")
print(f"✓ OOT ordenado: {oot_sorted}")

if not treino_sorted:
    print("\n⚠️  Ordenando dataset de treino...")
    df_treino = df_treino.sort_values('Timestamp').reset_index(drop=True)
    print("✅ Treino ordenado!")

if not oot_sorted:
    print("\n⚠️  Ordenando dataset de OOT...")
    df_oot = df_oot.sort_values('Timestamp').reset_index(drop=True)
    print("✅ OOT ordenado!")

print("\n✅ Validação concluída!")

## 4. Feature Engineering Simplificado - Criação de Safras e Features Temporais

Criamos features temporais básicas e a safra no formato YYYYMMDD.

In [ ]:
print("="*80)
print("FEATURE ENGINEERING COM PIPELINE MODULARIZADO")
print("="*80)

# ===== INSTANCIAÇÃO DO PIPELINE =====
# O pipeline garante execução na ordem correta (Ratio → Behavioral → Smurfing)
# e previne data leakage através de fit/transform apropriado

fe_pipeline = FeatureEngineeringPipeline(
    timestamp_col='Timestamp',
    account_col='From Account',
    amount_col='Amount Received',
    bank_col='Receiving Currency',      # Para mudança de banco
    country_col='From Bank',            # Para novo país
    # ===== VELOCITY FEATURES COMENTADAS PARA TESTE =====
    # ⚠️ DESABILITADO: velocity_windows descomente para ativar novamente
    # velocity_windows={                  # Janelas de velocidade
    #     '1h': '1H',                      # Última hora
    #     '24h': '24H',                    # Últimas 24 horas
    #     '7d': '7D'                       # Últimos 7 dias
    # },
    velocity_windows=None,              # 🚫 Velocity Features DESABILITADAS
    ratio_window='30D',                 # Comparação com histórico de 30 dias
    smurf_threshold=10000.0,            # Threshold para detecção de smurfing
    smurf_time_window='24H'             # Janela de tempo para smurfing
)

# ===== TREINAMENTO DO PIPELINE =====
print("\n🔄 Treinando pipeline no dataset de TREINO...")
fe_pipeline.fit(df_treino)

# ===== TRANSFORMAÇÃO DE TREINO =====
print("\n🔄 Aplicando transformações ao dataset de TREINO...")
df_treino_fe = fe_pipeline.transform(df_treino)

# ===== TRANSFORMAÇÃO DE OOT =====
# ⚠️ CRÍTICO: Usar MESMO pipeline (fit em treino apenas)
print("\n🔄 Aplicando transformações ao dataset de OOT (COM MESMO PIPELINE)...")
df_oot_fe = fe_pipeline.transform(df_oot)

# ===== ESTATÍSTICAS DE FEATURES GERADAS =====
original_cols = set(df_treino.columns)
new_features = [col for col in df_treino_fe.columns if col not in original_cols]

print(f"\n{'='*80}")
print("RESULTADO DO FEATURE ENGINEERING")
print("="*80)
print(f"\n📊 Shape:")
print(f"   Treino: {df_treino.shape} → {df_treino_fe.shape}")
print(f"   OOT: {df_oot.shape} → {df_oot_fe.shape}")

print(f"\n🎯 Total de novas features: {len(new_features)}")
print(f"\n📝 Categorias de features geradas:")

#velocity_features = [f for f in new_features if 'velocity' in f]
ratio_features = [f for f in new_features if 'ratio' in f]
behavioral_features = [f for f in new_features if 'behavioral' in f or 'bank_change' in f or 'time_since' in f or 'is_unusual' in f or 'is_new' in f]
smurf_features = [f for f in new_features if 'smurf' in f]

#print(f"   • Velocity Features ({len(velocity_features)}): {velocity_features if velocity_features else 'DESABILITADAS'}")
print(f"   • Ratio Features ({len(ratio_features)}): {ratio_features}")
print(f"   • Behavioral Features ({len(behavioral_features)}): {behavioral_features}")
print(f"   • Smurfing Features ({len(smurf_features)}): {smurf_features}")

# Salvar datasets com features
print(f"\n💾 Salvando datasets com features...")
df_treino_fe.to_csv(get_data_path('df_treino_fe.csv', 'processed'), index=False)
df_oot_fe.to_csv(get_data_path('df_oot_fe.csv', 'processed'), index=False)
print(f"✅ Datasets salvos em 'processed/'")

## 5. Análise das Features Criadas

Vamos analisar as distribuições das features temporais geradas.

In [ ]:
print("="*80)
print("ANÁLISE DAS FEATURES CRIADAS")
print("="*80)

# Exibir estatísticas das novas features
print("\n📊 Estatísticas das Features de Velocidade:")
velocity_cols = [col for col in df_treino_fe.columns if 'velocity' in col]
if velocity_cols:
    print(df_treino_fe[velocity_cols].describe())
else:
    print("Nenhuma feature de velocidade encontrada")

print("\n📊 Estatísticas das Features de Ratio:")
ratio_cols = [col for col in df_treino_fe.columns if 'ratio' in col]
if ratio_cols:
    print(df_treino_fe[ratio_cols].describe())
else:
    print("Nenhuma feature de ratio encontrada")

print("\n📊 Estatísticas das Features de Smurfing:")
smurf_cols = [col for col in df_treino_fe.columns if 'smurf' in col]
if smurf_cols:
    print(df_treino_fe[smurf_cols].describe())
else:
    print("Nenhuma feature de smurfing gerada")

# Correlação com target
print("\n📈 Correlação das Features com Target (Is Laundering):")
correlation_with_target = df_treino_fe[new_features + ['Is Laundering']].corr()['Is Laundering'].drop('Is Laundering').sort_values(ascending=False)
print(correlation_with_target.head(15))

print("\n✅ Análise concluída!")

In [ ]:
# Monitorar estabilidade das features entre treino e OOT
print("="*80)
print("MONITORAMENTO DE ESTABILIDADE (PSI)")
print("="*80)

# Calcular PSI para as novas features
print("\n🔍 Calculando PSI para features selecionadas...")

features_to_monitor = [
    col for col in df_treino_fe.columns 
    if 'velocity' in col or 'ratio' in col or 'smurf' in col
][:5]  # Primeiras 5 para visualização

psi_results = {}
for feature in features_to_monitor:
    if feature in df_treino_fe.columns:
        # Garantir que não há infinitos/NaNs
        train_scores = df_treino_fe[feature].replace([np.inf, -np.inf], np.nan).fillna(0).values
        oot_scores = df_oot_fe[feature].replace([np.inf, -np.inf], np.nan).fillna(0).values
        
        psi_result = BusinessMetrics.calculate_psi(train_scores, oot_scores, n_bins=10)
        psi_results[feature] = psi_result

print("\n📊 Resultado PSI (alvo < 0.1 para estabilidade):")
for feature, psi_dict in psi_results.items():
    psi_value = psi_dict['psi']
    category = psi_dict['psi_category']
    status = "✅" if psi_value < 0.1 else "⚠️"
    print(f"  {status} {feature}: PSI={psi_value:.4f} ({category})")

print("\n✅ Monitoramento concluído!")

## 6. Análise de Features de Negócio

Comparar taxa de lavagem em diferentes períodos (weekend, business hours, night).

## 7. Salvamento dos Dados com Features Simplificadas

Salvamos os datasets com features temporais para uso nos próximos notebooks.

In [ ]:
# Sumário final do Feature Engineering
print("="*80)
print("SUMÁRIO DO FEATURE ENGINEERING MODULARIZADO")
print("="*80)

print(f"\n✅ Datasets com features salvos:")
print(f"   - df_treino_fe.csv")
print(f"   - df_oot_fe.csv")

print(f"\n📊 Estatísticas Finais:")
print(f"   Features originais (treino): {df_treino.shape[1]}")
print(f"   Features novas: {len(new_features)}")
print(f"   Features totais (treino): {df_treino_fe.shape[1]}")

print(f"\n🔍 Distribuição de Features:")
print(f"   Velocity Features: {len([f for f in new_features if 'velocity' in f])}")
print(f"   Ratio Features: {len([f for f in new_features if 'ratio' in f])}")
print(f"   Behavioral Features: {len([f for f in new_features if 'behavioral' in f or 'bank_change' in f or 'time_since' in f])}")
print(f"   Smurfing Features: {len([f for f in new_features if 'smurf' in f])}")

print(f"\n⚠️ CRÍTICO - Zero Data Leakage:")
print(f"   ✅ Pipeline fit APENAS em treino")
print(f"   ✅ Transform aplicado em treino e OOT com MESMO pipeline")
print(f"   ✅ Ordenação temporal validada")
print(f"   ✅ Janelas deslizantes configuradas para excluir transação atual")

print(f"\n✅ Próximo: Notebook 06 - Pipeline de Transformação (Yeo-Johnson + Target Encoding)")

## 8. Sumário e Próximos Passos

### ✅ Realizações deste Notebook

1. ✅ Criação de **Safra (YYYYMMDD)** para análise temporal
2. ✅ Extração de **componentes temporais** (ano, mês, dia, hora, etc.)
3. ✅ Implementação de **features cíclicas** (sin/cos) para capturar periodicidade
4. ✅ Criação de **features de negócio** (weekend, business hours, night, etc.)
5. ✅ Análise de distribuição por safra, hora e dia da semana
6. ✅ Pipeline **simplificado e eficiente**

### 📊 Estatísticas Finais

- **Features originais**: {df_treino.shape[1]}
- **Features novas**: {len(new_features)}
- **Features totais**: {df_treino_fe.shape[1]}
- **Safras únicas (treino)**: {df_treino_fe['safra'].nunique()}
- **Safras únicas (OOT)**: {df_oot_fe['safra'].nunique()}

### 🔄 Próximos Passos

**Notebook 06**: Pipeline de Transformação Simplificado
- Encoding de variáveis categóricas
- Normalização de variáveis numéricas
- Sem necessidade de features de velocidade complexas

**Notebook 08**: Treinamento de Modelos
- Uso de safras para validação temporal
- Modelos mais simples e interpretáveis
- Foco em padrões temporais e comportamentais básicos

---

**Data**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}  
**Status**: ✅ Feature Engineering Simplificado Concluído